# Baseline dataset inspection

**Purpose:** Đây chỉ là bước kiểm tra, mô tả tổng quan dữ liệu và xây dựng mốc tham chiếu dựa trên lớp chiếm đa số; không thực hiện chia tập dữ liệu, huấn luyện, tinh chỉnh hay đánh giá mô hình


In [ ]:
# Purpose: Nhập các thư viện được sử dụng để tải dữ liệu, tổng hợp thông tin và tính toán các chỉ số đánh giá baseline
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)
pd.set_option('display.max_rows', 100)

In [30]:
# Purpose: Tìm vị trí tệp CSV bằng cách ưu tiên sử dụng các đường dẫn tương đối với thư mục dự án trước, sau đó sử dụng đường dẫn Windows được cung cấp làm phương án dự phòng. Tiếp theo, tải dữ liệu vào biến df
relative_candidates = [
    Path('DATA/raw/diabetes_012_health_indicators_BRFSS2015.csv'),
    Path('../../DATA/raw/diabetes_012_health_indicators_BRFSS2015.csv'),
    Path('../DATA/raw/diabetes_012_health_indicators_BRFSS2015.csv'),
]
windows_path = Path(r'C:\Users\Welcome\Desktop\ML\MAY_HOC_NHOM\Do_An_Cuoi_Ky\DATA\raw\diabetes_012_health_indicators_BRFSS2015.csv')
candidates = relative_candidates + [windows_path]
DATA_PATH = next((p.resolve() for p in candidates if p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('CSV not found. Checked: ' + ', '.join(map(str, candidates)))
print(f'Using: {DATA_PATH}')
df = pd.read_csv(DATA_PATH)
print(f'Loaded {len(df):,} rows and {df.shape[1]:,} columns.')

Using: C:\Users\Welcome\Desktop\ML\MAY_HOC_NHOM\Do_An_Cuoi_Ky\DATA\raw\diabetes_012_health_indicators_BRFSS2015.csv
Loaded 253,680 rows and 22 columns.


In [31]:
# Purpose: Hiển thị kích thước của tập dữ liệu và một mẫu dữ liệu nhỏ để có thể quan sát, kiểm tra trực quan dữ liệu đã được tải
print('Shape:', df.shape)
display(df.head())

Shape: (253680, 22)


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
0,0.0,1.0,1.0,1.0,40.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,5.0,18.0,15.0,1.0,0.0,9.0,4.0,3.0
1,0.0,0.0,0.0,0.0,25.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,0.0,0.0,0.0,7.0,6.0,1.0
2,0.0,1.0,1.0,1.0,28.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,1.0,5.0,30.0,30.0,1.0,0.0,9.0,4.0,8.0
3,0.0,1.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,11.0,3.0,6.0
4,0.0,1.0,1.0,1.0,24.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,3.0,0.0,0.0,0.0,11.0,5.0,4.0


In [32]:
# Purpose: Tổng hợp thông tin của từng cột, bao gồm kiểu dữ liệu, số lượng giá trị không bị thiếu, số lượng giá trị bị thiếu và số lượng giá trị phân biệt (không trùng lặp)
schema = pd.DataFrame({'dtype': df.dtypes.astype(str), 'non_null': df.notna().sum(), 'missing': df.isna().sum(), 'unique_values': df.nunique(dropna=True)})
display(schema)

,dtype,non_null,missing,unique_values
Diabetes_012,float64,253680,0,3
HighBP,float64,253680,0,2
HighChol,float64,253680,0,2
CholCheck,float64,253680,0,2
BMI,float64,253680,0,84
Smoker,float64,253680,0,2
Stroke,float64,253680,0,2
HeartDiseaseorAttack,float64,253680,0,2
PhysActivity,float64,253680,0,2
Fruits,float64,253680,0,2


In [33]:
# Purpose: Đếm số lượng mẫu thuộc từng lớp của biến Diabetes_012 và tính tỷ lệ của mỗi lớp trên tổng số mẫu của toàn bộ tập dữ liệu
TARGET = 'Diabetes_012'
if TARGET not in df.columns:
    raise KeyError(f'Required target {TARGET!r} not found')
class_counts = df[TARGET].value_counts(dropna=False).sort_index()
class_summary = pd.DataFrame({'count': class_counts, 'proportion': class_counts / len(df)})
class_summary['proportion_pct'] = class_summary['proportion'] * 100
display(class_summary)

,count,proportion,proportion_pct
Diabetes_012,,,
0.0,213703,0.842412,84.241170
1.0,4631,0.018255,1.825528
2.0,35346,0.139333,13.933302


In [34]:
# Purpose: Báo cáo tổng số ô dữ liệu bị thiếu trong toàn bộ tập dữ liệu và xác định những cột có chứa các giá trị bị thiếu.
missing = df.isna().sum().sort_values(ascending=False)
print('Total missing cells:', int(df.isna().sum().sum()))
display(missing[missing > 0].rename('missing_count').to_frame() if (missing > 0).any() else pd.DataFrame({'missing_count': pd.Series(dtype='int64')}))

Total missing cells: 0


,missing_count


In [35]:
# Purpose: Đếm chính xác số lượng các dòng dữ liệu bị trùng lặp hoàn toàn và báo cáo tỷ lệ của chúng trên tổng số dòng trong tập dữ liệu.
duplicate_count = int(df.duplicated().sum())
print(f'Exact duplicate rows: {duplicate_count:,} ({duplicate_count / len(df):.2%})')

Exact duplicate rows: 23,899 (9.42%)


In [36]:
# Purpose: Hiển thị các thống kê mô tả tiêu chuẩn cho tất cả các cột dữ liệu dạng số trong tập dữ liệu.
numeric_df = df.select_dtypes(include='number')
display(numeric_df.describe().T)

,count,mean,std,min,25%,50%,75%,max
Diabetes_012,253680.0,0.296921,0.698160,0.0,0.0,0.0,0.0,2.0
HighBP,253680.0,0.429001,0.494934,0.0,0.0,0.0,1.0,1.0
HighChol,253680.0,0.424121,0.494210,0.0,0.0,0.0,1.0,1.0
CholCheck,253680.0,0.962670,0.189571,0.0,1.0,1.0,1.0,1.0
BMI,253680.0,28.382364,6.608694,12.0,24.0,27.0,31.0,98.0
Smoker,253680.0,0.443169,0.496761,0.0,0.0,0.0,1.0,1.0
Stroke,253680.0,0.040571,0.197294,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,253680.0,0.094186,0.292087,0.0,0.0,0.0,0.0,1.0
PhysActivity,253680.0,0.756544,0.429169,0.0,1.0,1.0,1.0,1.0
Fruits,253680.0,0.634256,0.481639,0.0,0.0,1.0,1.0,1.0


In [ ]:
# Purpose:Dự đoán lớp mục tiêu xuất hiện thường xuyên nhất cho mọi dòng dữ liệu, sau đó tính toán độ chính xác (Accuracy) và các chỉ số đánh giá theo phương pháp One-vs-Rest (Precision, Recall và FNR) cho từng lớp, đồng thời xây dựng ma trận nhầm lẫn (Confusion Matrix)
class_counts = df[TARGET].value_counts().sort_index()
majority_class = class_counts.idxmax()
y_true = df[TARGET]
y_pred = pd.Series(majority_class, index=df.index, name='prediction')
print(f'Most common class: {majority_class:g} ({class_counts.loc[majority_class]:,} rows)')
print(f'Prediction rule: predict {majority_class:g} for every row')

accuracy = (y_true == y_pred).mean()
print(f'Accuracy: {accuracy:.4%}')

# Các chỉ số đánh giá theo phương pháp One-vs-Rest cho từng lớp của biến mục tiêu. Tỷ lệ âm tính giả (FNR) được tính theo công thức
metric_rows = []
for label in class_counts.index:
    actual_positive = y_true.eq(label)
    predicted_positive = y_pred.eq(label)
    tp = int((actual_positive & predicted_positive).sum())
    fp = int((~actual_positive & predicted_positive).sum())
    fn = int((actual_positive & ~predicted_positive).sum())
    tn = int((~actual_positive & ~predicted_positive).sum())
    precision = tp / (tp + fp) if tp + fp else np.nan
    recall = tp / (tp + fn) if tp + fn else np.nan
    fnr = fn / (tp + fn) if tp + fn else np.nan
    metric_rows.append({'class': label, 'support': int(actual_positive.sum()), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn, 'precision': precision, 'recall': recall, 'FNR': fnr})
class_metrics = pd.DataFrame(metric_rows).set_index('class')
display(class_metrics.style.format({'precision': '{:.2%}', 'recall': '{:.2%}', 'FNR': '{:.2%}'}))

# Ma trận nhầm lẫn (Confusion Matrix): Các hàng biểu thị lớp thực tế, các cột biểu thị lớp được dự đoán.
confusion = pd.crosstab(y_true, y_pred, rownames=['Actual'], colnames=['Predicted'], dropna=False)
display(confusion)

Most common class: 0 (213,703 rows)
Prediction rule: predict 0 for every row
Accuracy: 84.2412%


,support,TP,FP,FN,TN,precision,recall,FNR
class,,,,,,,,
0.000000,213703,213703,39977,0,0,84.24%,100.00%,0.00%
1.000000,4631,0,0,4631,249049,nan%,0.00%,100.00%
2.000000,35346,0,0,35346,218334,nan%,0.00%,100.00%


Predicted,0.0
Actual,
0.0,213703
1.0,4631
2.0,35346


## Baseline observations

**Purpose:** Các chỉ số được tính toán trên toàn bộ tập dữ liệu đã được sử dụng để lựa chọn lớp chiếm đa số. Vì vậy, hãy xem chúng là các giá trị tham chiếu mang tính mô tả, thay vì kết quả đánh giá trên một tập dữ liệu kiểm tra độc lập (Held-out Performance).


In [ ]:
# Purpose: In ra lớp chiếm đa số đã được xác định cùng với các chỉ số đánh giá Baseline
print(f'Majority class: {majority_class:g} ({class_counts.loc[majority_class]:,}/{len(df):,}, {class_counts.loc[majority_class]/len(df):.2%})')
print(f'Accuracy: {accuracy:.2%}')
for label, row in class_metrics.iterrows():
    precision_text = 'undefined (no positive predictions)' if pd.isna(row['precision']) else f"{row['precision']:.2%}"
    print(f"Class {label:g}: precision {precision_text}; recall {row['recall']:.2%}; FNR {row['FNR']:.2%}")

print('Observation: predicting only the majority class gives strong overall accuracy, but misses every example in any class it never predicts. Use per-class recall/FNR alongside accuracy.')


Majority class: 0 (213,703/253,680, 84.24%)
Accuracy: 84.24%
Class 0: precision 84.24%; recall 100.00%; FNR 0.00%
Class 1: precision undefined (no positive predictions); recall 0.00%; FNR 100.00%
Class 2: precision undefined (no positive predictions); recall 0.00%; FNR 100.00%
Observation: predicting only the majority class gives strong overall accuracy, but misses every example in any class it never predicts. Use per-class recall/FNR alongside accuracy.
